# BB test on tick data

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
date_ = "16APR2026"
file_name = "NIFTY 50.xlsx"
file_path = Path(fr"D:\Study\Programs\trading\assets\logs\{date_}\extracted_symbols\{file_name}")
nifty_all = pd.read_excel(file_path)

In [3]:
nifty_all.columns

Index(['instrument_token', 'symbol', 'exchange_timestamp', 'local_time',
       'last_trade_time', 'last_price', 'option_CE_PE', 'option_type',
       'strike', 'ohlc', 'change', 'tradable', 'mode'],
      dtype='str')

In [4]:
nifty_all.head()

,instrument_token,symbol,exchange_timestamp,local_time,last_trade_time,last_price,option_CE_PE,option_type,strike,ohlc,change,tradable,mode
0,256265,NIFTY 50,NaN,2026-04-16 09:10:05.245,NaN,24385.2,NaN,index,NaN,"{'high': 24385.2, 'low': 24385.2, 'open': 2438...",0.635129,False,full
1,256265,NIFTY 50,NaN,2026-04-16 09:10:05.245,NaN,24385.2,NaN,index,NaN,"{'high': 24385.2, 'low': 24385.2, 'open': 2438...",0.635129,False,full
2,256265,NIFTY 50,NaN,2026-04-16 09:10:05.245,NaN,24385.2,NaN,index,NaN,"{'high': 24385.2, 'low': 24385.2, 'open': 2438...",0.635129,False,full
3,256265,NIFTY 50,NaN,2026-04-16 09:10:05.245,NaN,24385.2,NaN,index,NaN,"{'high': 24385.2, 'low': 24385.2, 'open': 2438...",0.635129,False,full
4,256265,NIFTY 50,NaN,2026-04-16 09:10:05.245,NaN,24385.2,NaN,index,NaN,"{'high': 24385.2, 'low': 24385.2, 'open': 2438...",0.635129,False,full


In [5]:
type(nifty_all.iloc[0]["local_time"])

pandas.Timestamp

In [6]:
nifty_all.iloc[4]["local_time"].hour, nifty_all.iloc[4]["local_time"].minute, nifty_all.iloc[4]["local_time"].second

(9, 10, 5)

In [7]:
# nifty_all["local_time"] = pd.to_datetime(nifty_all["local_time"])
# nifty_all["hhmmss"] = nifty_all["local_time"].dt.strftime("%H%M%S")
# nifty_all["hhmm"] = nifty_all["local_time"].dt.strftime("%H%M")

In [8]:
nifty_all = nifty_all.sort_values("local_time")
nifty_all["minute"] = nifty_all["local_time"].dt.floor("min")
grp = nifty_all.groupby("minute")

nifty_all["open"]  = grp["last_price"].transform("first")
nifty_all["high"]  = grp["last_price"].cummax()
nifty_all["low"]   = grp["last_price"].cummin()
nifty_all["close"] = nifty_all["last_price"]

In [9]:
# nifty_all.to_excel("nifty_all_dev.xlsx")

In [10]:
import pandas as pd
import numpy as np
import pandas_ta as ta

# =========================================================
# INDICATORS (UNCHANGED)
# =========================================================

def tradingview_bb(close, length=20, mult=2.0):
    basis = close.rolling(length).mean()
    std = close.rolling(length).std(ddof=0)
    dev = mult * std
    upper = basis + dev
    lower = basis - dev
    return basis, upper, lower


def tradingview_roc(close, length=10):
    prev = close.shift(length)
    roc = 100 * (close / prev - 1)
    return roc


def compute_signals(df,
                    bb_length=20,
                    bb_mult=2.0,
                    dmi_length=14,
                    roc_length=10):

    df = df.copy()
    df = df.sort_values('date').reset_index(drop=True)

    # BB
    df['basis'], df['upper_bb'], df['lower_bb'] = tradingview_bb(
        df['close'], bb_length, bb_mult
    )

    # DMI
    dmi = ta.adx(df['high'], df['low'], df['close'], length=dmi_length)
    df['plus_di']  = dmi[[c for c in dmi.columns if "DMP" in c][0]]
    df['minus_di'] = dmi[[c for c in dmi.columns if "DMN" in c][0]]
    df['adx']      = dmi[[c for c in dmi.columns if "ADX" in c][0]]

    # ROC
    df['roc'] = tradingview_roc(df['close'], roc_length)

    return df


# =========================================================
# MAIN ENGINE
# =========================================================

def generate_tick_signals(tick_df):

    tick_df = tick_df.copy()
    tick_df = tick_df.sort_values("local_time")

    # -----------------------------------------------------
    # 1. BUILD MINUTE CANDLES
    # -----------------------------------------------------
    minute_df = tick_df.resample("1min", on="local_time").agg({
        "last_price": ["first", "max", "min", "last"]
    })

    minute_df.columns = ["open", "high", "low", "close"]
    minute_df = minute_df.reset_index().rename(columns={"local_time": "date"})

    # -----------------------------------------------------
    # 2. COMPUTE INDICATORS (MINUTE ONLY)
    # -----------------------------------------------------
    minute_df = compute_signals(minute_df)

    # -----------------------------------------------------
    # 3. MAP PREVIOUS MINUTE INDICATORS TO TICKS
    # -----------------------------------------------------
    tick_df["minute"] = tick_df["local_time"].dt.floor("min")

    ind_cols = ["basis","upper_bb","lower_bb","plus_di","minus_di","adx","roc"]

    tick_df = tick_df.merge(
        minute_df[["date"] + ind_cols],
        left_on="minute",
        right_on="date",
        how="left"
    )

    # shift → use ONLY completed candle
    for col in ind_cols:
        tick_df[col] = tick_df[col].shift(1)

    # -----------------------------------------------------
    # 4. BUILD INTRA-MINUTE OHLC (TICK EVOLUTION)
    # -----------------------------------------------------
    grp = tick_df.groupby("minute")

    tick_df["open_tick"]  = grp["last_price"].transform("first")
    tick_df["high_tick"]  = grp["last_price"].cummax()
    tick_df["low_tick"]   = grp["last_price"].cummin()
    tick_df["close_tick"] = tick_df["last_price"]

    # -----------------------------------------------------
    # 5. SIGNAL LOGIC (USING TICKS + MINUTE INDICATORS)
    # -----------------------------------------------------
    tick_df["long_signal"] = (
        (tick_df["close_tick"].shift(1) < tick_df["lower_bb"].shift(1)) &
        (tick_df["open_tick"] < tick_df["lower_bb"]) &
        (tick_df["minus_di"] < 0.95 * tick_df["minus_di"].shift(1)) &
        (tick_df["roc"] > tick_df["roc"].shift(1) + 0.01)
    )

    tick_df["short_signal"] = (
        (tick_df["close_tick"].shift(1) > tick_df["upper_bb"].shift(1)) &
        (tick_df["open_tick"] > tick_df["upper_bb"]) &
        (tick_df["plus_di"] < 0.95 * tick_df["plus_di"].shift(1)) &
        (tick_df["roc"] < tick_df["roc"].shift(1) - 0.01)
    )

    tick_df["signal"] = np.where(
        tick_df["long_signal"], "BUY",
        np.where(tick_df["short_signal"], "SELL", None)
    )

    # -----------------------------------------------------
    # 6. CAPTURE FIRST SIGNAL PER MINUTE
    # -----------------------------------------------------
    tick_df["signal_triggered"] = (
        tick_df["signal"].notna() &
        tick_df["signal"].ne(tick_df["signal"].shift(1))
    )

    # -----------------------------------------------------
    # 7. CAPTURE SECOND INSIDE MINUTE
    # -----------------------------------------------------
    tick_df["second"] = tick_df["local_time"].dt.second

    # OPTIONAL: only keep first signal in each minute
    tick_df["first_signal_in_minute"] = (
        tick_df.groupby("minute")["signal_triggered"]
        .transform(lambda x: x & ~x.shift(1).fillna(False))
    )

    return tick_df

In [11]:
nifty_all = generate_tick_signals(nifty_all)

In [12]:
from pathlib import Path
import os

path_obj = Path(file_name)
new_file_name = path_obj.with_stem(f"{path_obj.stem}_signal").name
file_path = Path(fr"D:\Study\Programs\trading\assets\logs\{date_}\signals\{new_file_name}")
os.makedirs(os.path.dirname(file_path), exist_ok=True)
nifty_all.to_excel(file_path)

In [13]:
new_file_name

'NIFTY 50_signal.xlsx'